In [1]:
"""
Skin Disease Detection Training Pipeline
Dataset: HAM10000
Goal: Automatically select the best model and fine-tune it
"""

import os
import numpy as np
import pandas as pd
from glob import glob
from PIL import Image
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import EfficientNetB3, EfficientNetB4, DenseNet121, ResNet50
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [15]:
# SETTINGS

IMAGE_FOLDERS = ["./HAM10000_images_part1", "./HAM10000_images_part2"]
METADATA_CSV = "./HAM10000_metadata.csv"
FINAL_MODEL_PATH = "saved_models/final_skin_model.keras"
TARGET_SIZE = 224  # input size for models

In [17]:
# 1. Load images and labels

def load_dataset():
    metadata = pd.read_csv(METADATA_CSV)
    metadata = metadata[['image_id', 'dx']]  # only id and label
    imgs, labels = [], []

    for _, row in metadata.iterrows():
        img_id, label = row['image_id'], row['dx']
        found = False
        for folder in IMAGE_FOLDERS:
            img_path = os.path.join(folder, f"{img_id}.jpg")
            if os.path.exists(img_path):
                img = Image.open(img_path).convert('RGB').resize((TARGET_SIZE, TARGET_SIZE))
                imgs.append(np.array(img))
                labels.append(label)
                found = True
                break
        if not found:
            print("Warning: missing image:", img_id)

    X = np.array(imgs, dtype=np.float32) / 255.0
    y = np.array(labels)
    return X, y

In [19]:
# 2. Encode class labels

def encode_labels(y):
    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)
    class_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
    return y_encoded, class_mapping, encoder

In [21]:
# 3. Standardize features

def standardize_data(X_train, X_val):
    h, w, c = X_train.shape[1:]
    scaler = StandardScaler()
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_train_scaled = scaler.fit_transform(X_train_flat).reshape(-1, h, w, c)
    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    X_val_scaled = scaler.transform(X_val_flat).reshape(-1, h, w, c)
    return X_train_scaled, X_val_scaled

In [23]:
# 4. Build candidate models

def create_model(arch_name, num_classes):
    if arch_name == 'EfficientNetB3':
        base = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(TARGET_SIZE, TARGET_SIZE, 3))
    elif arch_name == 'EfficientNetB4':
        base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=(TARGET_SIZE, TARGET_SIZE, 3))
    elif arch_name == 'DenseNet121':
        base = DenseNet121(weights='imagenet', include_top=False, input_shape=(TARGET_SIZE, TARGET_SIZE, 3))
    elif arch_name == 'ResNet50':
        base = ResNet50(weights='imagenet', include_top=False, input_shape=(TARGET_SIZE, TARGET_SIZE, 3))
    else:
        raise ValueError("Unknown architecture")

    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=outputs)
    return model, base

In [25]:
# 5. Train & select best model

def train_and_select():
    print("Loading dataset...")
    X, y_raw = load_dataset()
    y, class_map, label_enc = encode_labels(y_raw)
    num_classes = len(class_map)

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    X_train, X_val = standardize_data(X_train, X_val)

    class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_weights = dict(enumerate(class_weights_arr))
    print("Class weights:", class_weights)

    data_aug = ImageDataGenerator(
        rotation_range=25,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.2,
        shear_range=0.1,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.7,1.3],
        fill_mode='nearest'
    )
    data_aug.fit(X_train)

    candidate_archs = ['EfficientNetB3', 'EfficientNetB4', 'DenseNet121', 'ResNet50']
    top_acc = 0
    best_model = None
    best_name = None

    for arch in candidate_archs:
        print(f"\nTraining {arch}...")
        model, base = create_model(arch, num_classes)
        model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.2)
        ]
        history = model.fit(data_aug.flow(X_train, y_train, batch_size=32),
                            validation_data=(X_val, y_val),
                            epochs=10,
                            class_weight=class_weights,
                            callbacks=callbacks,
                            verbose=2)
        val_acc = max(history.history['val_accuracy'])
        print(f"{arch} validation accuracy: {val_acc:.4f}")

        if val_acc > top_acc:
            top_acc = val_acc
            best_model = model
            best_name = arch

    print(f"\n Selected architecture: {best_name} with accuracy {top_acc:.4f}")

    # Fine-tune best model
    for layer in best_model.layers:
        layer.trainable = True
    best_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])
    best_model.fit(data_aug.flow(X_train, y_train, batch_size=16),
                   validation_data=(X_val, y_val),
                   epochs=20,
                   class_weight=class_weights,
                   callbacks=callbacks)

    best_model.save(FINAL_MODEL_PATH)
    print(f"Model saved at {FINAL_MODEL_PATH}")
    return best_model, label_enc, X_val, y_val


if __name__ == "__main__":
    train_and_select()


Loading dataset...


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.